# HC-FALCON: tính embedding trên GPU (Colab / Kaggle)

Máy local không có GPU, nên mọi bước embedding chạy ở đây rồi tải kết quả về.

1. **Runtime → Change runtime type → GPU** (T4 là đủ).
2. Chạy lần lượt các ô của chế độ bạn cần.
3. Tải file zip ở ô cuối, giải nén vào `data/embeddings/` trong repo local.

| Chế độ | Khi nào | Kết quả |
|---|---|---|
| **(a) Tái lập FALCON** trên Defects4J | Tuần 2 | `d4j/<Project>/unixcoder_test_embeddings.npy`, thứ tự do code FALCON gốc sinh ra, APFD của artifact |
| **(b) LRTS** (masked mean-pool từng method) | Tuần 4, 7 | `lrts/<project>/unixcoder_methods.npz` |

Trên Kaggle: bật GPU trong *Settings → Accelerator*, đổi `BASE` thành `/kaggle/working`, tải zip ở tab *Output*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## Chế độ (a): tái lập FALCON trên 6 dự án Defects4J

Dùng **nguyên code của artifact FALCON** (`py/embedding_generator.py`, `py/submodtcp.py`, `py/metric.py`) để embedding và thứ tự trùng với cách tác giả làm. Lưu ý: generator của FALCON lấy trung bình cả token padding (không dùng attention mask), batch 16, max 512 token. Ta giữ nguyên để tái lập; chế độ (b) sẽ dùng masked mean-pool.

In [ ]:
BASE = '/content'   # Kaggle: '/kaggle/working'
PROJECTS = ['Chart', 'Closure', 'Lang', 'Math', 'Mockito', 'Time']
MODEL = 'unixcoder'
FALCON_REPO = 'https://github.com/anonymous-ing/falcon-icst-2026.git'

In [ ]:
import os, subprocess
os.chdir(BASE)
if not os.path.isdir('falcon-icst-2026'):
    subprocess.run(['git', 'clone', '--depth', '1', FALCON_REPO], check=True)
FALCON_COMMIT = subprocess.run(['git', '-C', 'falcon-icst-2026', 'rev-parse', 'HEAD'],
                               capture_output=True, text=True, check=True).stdout.strip()
print('FALCON artifact commit:', FALCON_COMMIT)

In [ ]:
# Sinh embedding bằng script gốc của FALCON (mỗi dự án vài chục giây trên T4)
os.chdir(f'{BASE}/falcon-icst-2026')
for P in PROJECTS:
    print('=' * 20, P)
    r = subprocess.run(['python', 'py/embedding_generator.py', '--project', P,
                        '--models', MODEL, '--device', 'cuda'], capture_output=True, text=True)
    print(r.stdout[-600:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'embedding failed for {P}')

In [ ]:
# Chạy FacilityLocation.lazy_greedy của artifact, lưu thứ tự + APFD để so với bản cài đặt local
import sys, json, time, shutil
import numpy as np
sys.path.insert(0, 'py')
import submodtcp, metric

EXPORT = f'{BASE}/export/d4j'
summary = []
for P in PROJECTS:
    emb_file = f'input/{P}/embeddings/{MODEL}_test_embeddings.npy'
    emb = np.load(emb_file)
    t0 = time.perf_counter()
    fl = submodtcp.FacilityLocation(source_embeddings=emb, metric='cosine', device='cuda')
    order, _ = fl.lazy_greedy(budget=emb.shape[0])
    elapsed = time.perf_counter() - t0
    apfd = metric.apfd(order, f'input/{P}/fault_matrix.pickle')
    out = f'{EXPORT}/{P}'
    os.makedirs(out, exist_ok=True)
    shutil.copy(emb_file, out)
    shutil.copy(f'input/{P}/embeddings/{MODEL}_timing.json', out)
    np.save(f'{out}/falcon_artifact_order.npy', np.asarray(order, dtype=np.int64))
    rec = dict(project=P, n=int(emb.shape[0]), dim=int(emb.shape[1]), apfd=float(apfd),
               prioritization_time_s=round(elapsed, 3), device='cuda',
               gpu=torch.cuda.get_device_name(0), falcon_commit=FALCON_COMMIT)
    with open(f'{out}/falcon_artifact_result.json', 'w') as fh:
        json.dump(rec, fh, indent=2)
    summary.append(rec)
    print(rec)

print('median APFD:', float(np.median([r['apfd'] for r in summary])), '(paper: 0.731)')

In [ ]:
# Đóng gói và tải về. Local: giải nén vào <repo>/data/embeddings/  -> data/embeddings/d4j/<Project>/...
archive = shutil.make_archive(f'{BASE}/d4j_export', 'zip', f'{BASE}/export')
print(archive, round(os.path.getsize(archive) / 1e6, 1), 'MB')
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Không phải Colab: tải file ở tab Output (Kaggle).')

## Chế độ (b): LRTS — embed các method test của một dự án

Đầu vào là một hoặc nhiều corpus `<project>_methods.jsonl.gz`, sinh trên máy local bằng
`python -m hcfalcon.lrts.export_blobs --project <project>` (nằm ở `data/lrts/corpus/`). Mỗi dòng là một method
duy nhất `{"id", "text"}`. Method giống nhau giữa các file và các phiên bản chỉ embed một lần.

Mô hình `microsoft/unixcoder-base`, cắt 512 token, **masked mean-pool** (không tính token padding, khác code
FALCON gốc), rồi chuẩn hóa L2. Vector của test class được ghép ở máy local (`hcfalcon.lrts.embeddings`).

Kết quả: `lrts/<project>/unixcoder_methods.npz` (`ids`, `vectors` float16) và `unixcoder_methods_timing.json`
(dùng cho RQ4). Tải `lrts_export.zip` về, giải nén vào `data/embeddings/` → `data/embeddings/lrts/<project>/...`.

Chỉ cần chạy ô kiểm tra GPU ở đầu notebook, không cần chạy các ô của chế độ (a).

In [ ]:
BASE = '/content'          # Kaggle: '/kaggle/working'
PROJECTS = ['jackrabbit-oak', 'hbase']   # mỗi dự án một file <project>_methods.jsonl.gz
MODEL_NAME = 'microsoft/unixcoder-base'
MAX_LENGTH = 512
BATCH_SIZE = 64
CORPUS = {p: f'{BASE}/{p}_methods.jsonl.gz' for p in PROJECTS}

In [ ]:
# Colab: chọn cùng lúc các file data/lrts/corpus/<project>_methods.jsonl.gz trên máy bạn.
# Kaggle: thêm các file làm Dataset rồi sửa CORPUS cho đúng đường dẫn /kaggle/input/...
import os
missing = [p for p in PROJECTS if not os.path.exists(CORPUS[p])]
if missing:
    from google.colab import files
    for name in files.upload():
        for p in PROJECTS:
            if name.startswith(p + '_methods'):
                os.replace(name, CORPUS[p])
for p in PROJECTS:
    if os.path.exists(CORPUS[p]):
        print(p, CORPUS[p], round(os.path.getsize(CORPUS[p]) / 1e6, 2), 'MB')
    else:
        print(p, 'MISSING: upload', os.path.basename(CORPUS[p]))

In [ ]:
import gzip, json, time
import numpy as np
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
if device == 'cuda':
    model = model.half()

def embed(texts):
    order = np.argsort([-len(t) for t in texts])   # similar lengths per batch -> little padding
    vectors = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    n_tokens = n_truncated = 0
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for s in range(0, len(texts), BATCH_SIZE):
            idx = order[s:s + BATCH_SIZE]
            enc = tokenizer([texts[i] for i in idx], padding=True, truncation=True, max_length=MAX_LENGTH,
                            return_tensors='pt').to(device)
            hidden = model(**enc).last_hidden_state.float()
            mask = enc['attention_mask'].unsqueeze(-1).float()
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            vectors[idx] = torch.nn.functional.normalize(pooled, dim=-1).cpu().numpy()
            n_tokens += int(enc['attention_mask'].sum())
            n_truncated += int((enc['attention_mask'].sum(1) == MAX_LENGTH).sum())
    if device == 'cuda':
        torch.cuda.synchronize()
    return vectors, n_tokens, n_truncated, time.perf_counter() - t0

results = {}
for p in PROJECTS:
    with gzip.open(CORPUS[p], 'rt', encoding='utf-8') as fh:
        rows = [json.loads(line) for line in fh]
    ids, texts = [r['id'] for r in rows], [r['text'] for r in rows]
    vectors, n_tokens, n_truncated, elapsed = embed(texts)
    results[p] = (ids, vectors, n_tokens, n_truncated, elapsed)
    print(f'{p}: {len(texts)} methods, {elapsed:.1f}s, {len(texts) / elapsed:.0f} methods/s, truncated: {n_truncated}')

In [ ]:
import shutil
shutil.rmtree(f'{BASE}/export_lrts', ignore_errors=True)
for p, (ids, vectors, n_tokens, n_truncated, elapsed) in results.items():
    out = f'{BASE}/export_lrts/lrts/{p}'
    os.makedirs(out, exist_ok=True)
    np.savez_compressed(f'{out}/unixcoder_methods.npz', ids=np.array(ids), vectors=vectors.astype(np.float16))
    timing = dict(project=p, model=MODEL_NAME, n_methods=len(ids), n_tokens=n_tokens, n_truncated=n_truncated,
                  max_length=MAX_LENGTH, batch_size=BATCH_SIZE, pooling='masked_mean', seconds=round(elapsed, 2),
                  device=device, gpu=torch.cuda.get_device_name(0) if device == 'cuda' else None,
                  torch=torch.__version__, transformers=transformers.__version__)
    with open(f'{out}/unixcoder_methods_timing.json', 'w') as fh:
        json.dump(timing, fh, indent=2)
    print(timing)
archive = shutil.make_archive(f'{BASE}/lrts_export', 'zip', f'{BASE}/export_lrts')
print(archive, round(os.path.getsize(archive) / 1e6, 1), 'MB')
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Không phải Colab: tải file ở tab Output (Kaggle).')